# Proyecto Pareja 2: MD de System_2A y System_2B en GPU
Mismo protocolo para los dos sistemas (`B_dinamica/mdp/`). Ejecuta las celdas **en orden**.
Antes de empezar: *Entorno de ejecución > Cambiar tipo de entorno > A100*.

Si Colab se desconecta, vuelve a ejecutar desde la celda 1: las etapas terminadas se saltan y la producción retoma desde su checkpoint.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi

## 2. Instalar conda
El entorno se reinicia solo al terminar y aparece un aviso de sesión caída: es normal. Sigue con la celda 3.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 3. Instalar GROMACS con CUDA

In [ ]:
!mamba install -y -c conda-forge -c bioconda --no-pin gromacs=*=*cuda* cuda-version=12.6
!ln -sf /usr/local/bin/gmx_mpi /usr/bin/gmx
!ln -sf /usr/local/bin/gmx_mpi /usr/local/bin/gmx
!gmx --version | grep -E "GROMACS version|GPU support"

## 4. Montar Drive y crear las carpetas
Crea `MyDrive/proyecto_2/` con la misma estructura del repositorio. La primera vez pide subir `proyecto_2_colab.zip`.

In [ ]:
import os, subprocess
from google.colab import drive, files
drive.mount('/content/drive')

RAIZ = '/content/drive/MyDrive/proyecto_2'
for carpeta in ['01_estructuras', 'B_dinamica/mdp', 'B_dinamica/colab', 'B_dinamica/2A', 'B_dinamica/2B',
                'B_dinamica/figuras', 'C_analisis/datos', 'C_analisis/figuras', 'scripts']:
    os.makedirs(f'{RAIZ}/{carpeta}', exist_ok=True)

if not os.path.exists(f'{RAIZ}/01_estructuras/System_2A.pdb'):
    subido = files.upload()
    nombre = list(subido)[0]
    subprocess.run(['unzip', '-o', nombre, '-d', RAIZ], check=True)
!ls -R "/content/drive/MyDrive/proyecto_2" | head -40

## 5. Dinámica de 2A (B1–B8)

In [ ]:
!bash "/content/drive/MyDrive/proyecto_2/B_dinamica/colab/correr_sistema.sh" 2A "/content/drive/MyDrive/proyecto_2" 2>&1 | grep -E "^===|Total charge|Performance|Fatal|ERROR|Last frame|Coords"

## 6. Dinámica de 2B (B1–B8)

In [ ]:
!bash "/content/drive/MyDrive/proyecto_2/B_dinamica/colab/correr_sistema.sh" 2B "/content/drive/MyDrive/proyecto_2" 2>&1 | grep -E "^===|Total charge|Performance|Fatal|ERROR|Last frame|Coords"

## 7. Verificaciones y análisis (partes B y C)
Guarda los `.xvg` y `resumen_verificaciones.txt` en `C_analisis/datos/<sistema>/`.

In [ ]:
!bash "/content/drive/MyDrive/proyecto_2/B_dinamica/colab/analizar_sistema.sh" 2A "/content/drive/MyDrive/proyecto_2" 2>&1 | tail -3
!bash "/content/drive/MyDrive/proyecto_2/B_dinamica/colab/analizar_sistema.sh" 2B "/content/drive/MyDrive/proyecto_2" 2>&1 | tail -3
!cat "/content/drive/MyDrive/proyecto_2/C_analisis/datos/2A/resumen_verificaciones.txt"
!cat "/content/drive/MyDrive/proyecto_2/C_analisis/datos/2B/resumen_verificaciones.txt"

## 8. Gráficas 2A vs 2B

In [ ]:
!mamba install -y -q -c conda-forge matplotlib-base numpy < /dev/null
import os, subprocess
RAIZ = '/content/drive/MyDrive/proyecto_2'
D = f'{RAIZ}/C_analisis/datos'
for x in ['C2_rmsd_complejo', 'C2_rmsd_nanobody', 'C2_rmsd_antigeno', 'C2_rmsd_nanobody_sobre_antigeno',
          'C3_contactos_4A', 'C3_puentes_H', 'C3_distancia_minima', 'C4_salino_Arg_Glu111',
          'C4_Tyr27_Gln60', 'B8_temperatura', 'B8_densidad']:
    subprocess.run(['python3', f'{RAIZ}/scripts/graficar_xvg.py', f'{D}/2A/{x}.xvg', f'{D}/2B/{x}.xvg',
                    '--etiquetas', '2A', '2B', '-o', f'{RAIZ}/C_analisis/figuras/{x}_2A_vs_2B.png'],
                   check=True, env={**os.environ, 'MPLBACKEND': 'Agg'})
!ls "/content/drive/MyDrive/proyecto_2/C_analisis/figuras"

## 9. Listo
Todo queda en `MyDrive/proyecto_2/`. Descarga `C_analisis/` y, de cada sistema, `proteina.tpr`, `proteina.xtc`, `proteina.ndx`, `md.log` y `topol.top`.